# Notebook 05 — Objetivo Específico 3 y Pruebas de Hipótesis: Validación Olofsson / Collect Earth Online (100% en SEPAL)

- **Instancia SEPAL recomendada:** `t2` o `m2` (cómputo estadístico local en segundos, cero GEE).
- **Flujo:**
  1. Sube el CSV de diseño `outputs/muestras/diseno_muestral_area2_para_CEO_<region>.csv` a **Collect Earth Online (CEO)** e interpreta visualmente cada punto sobre mosaicos **Sentinel-2 y Planet** (o usa un diseño generado en la app AREA2 de GEE sobre `mapa_estratos_area2_...`).
  2. Exporta el CSV interpretado desde CEO a `outputs/muestras/ceo_validacion_<region>.csv`.
  3. Este notebook cruza en **< 1 segundo** los puntos validados en CEO (`EPSG:4326`) contra tus **4 GeoTIFFs locales en SEPAL** (`MS`, `SAR`, `MS_SAR_PCA`, `MS_SAR_FULL`) y calcula **OA, PU, PP, F1-Score ± IC 95%**, **Área ajustada insesgada (ha ± IC 95%)** y las pruebas formales **McNemar, Z pareada ponderada por diseño y Z de Kappa**.

In [ ]:
# 1. Configuracion e ingesta del archivo CSV exportado desde Collect Earth Online (CEO)
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config_utils import ensure_output_dirs, get_class_maps, load_config
from src.local_rf import cross_points_with_local_maps
from src.metrics import (
    kappa_z_test,
    mcnemar_test,
    stehman_generalized_accuracy,
    stratified_paired_z_test,
)
from src.plotting import plot_confusion_matrix, plot_model_comparison

# >>> CAMBIAR AQUI ENTRE "PACIFICO" Y "CARIBE" <<<
REGION = "PACIFICO"
reg_l = REGION.lower()

cfg = load_config()
dirs = ensure_output_dirs(cfg)
ver = cfg["proyecto"]["version_pipeline"].lower()
val_cfg = cfg["validacion_olofsson"]

CLASES_NOMBRES, CLASES_CORTOS, CLASES_COLORES = get_class_maps(cfg)
CLASS_IDS = sorted(CLASES_NOMBRES.keys())

# Ruta esperada del CSV exportado desde Collect Earth Online
CEO_CSV_PATH = dirs["muestras"] / f"ceo_validacion_{reg_l}.csv"

# Columnas habituales en CEO + AREA2 / diseno local (ajusta segun tu formulario CEO)
COL_LON = "lon"         # en CEO suele ser 'lon' o 'LON' o 'center_lon'
COL_LAT = "lat"         # en CEO suele ser 'lat' o 'LAT' o 'center_lat'
COL_STRATUM = "stratum" # o 'stratum_id'
COL_REF = "ref_class"   # nombre de la pregunta de cobertura en tu proyecto CEO

# Mapeo de respuestas de CEO a los IDs del Protocolo (1..5)
MAPEO_ETIQUETAS_CEO = {
    1: 1, "1": 1, "Mangle": 1, "Manglar": 1, "mangle": 1,
    2: 2, "2": 2, "Bosque latifoliado": 2, "Latifoliado": 2, "Bosque": 2,
    3: 3, "3": 3, "Agua": 3, "agua": 3,
    4: 4, "4": 4, "Infraestructura urbana": 4, "Urbano": 4, "Infraestructura": 4,
    5: 5, "5": 5, "Pastos/Ganadería": 5, "Pastos/Ganaderia": 5, "Pastos": 5, "Ganaderia": 5,
}

# Indica que mapa se uso para estratificar las muestras:
#   "ACUERDO_DESACUERDO" (6 estratos) o "MS_SAR_PCA" (5 clases)
TIPO_ESTRATO_DISENO = "ACUERDO_DESACUERDO"

areas_json = json.loads((dirs["tablas"] / f"areas_estratos_{reg_l}.json").read_text(encoding="utf-8"))
if TIPO_ESTRATO_DISENO == "ACUERDO_DESACUERDO":
    STRATUM_AREAS_HA = {int(k): float(v) for k, v in areas_json["areas_estratos_acuerdo_desacuerdo_ha"].items()}
else:
    STRATUM_AREAS_HA = {int(k): float(v) for k, v in areas_json["areas_modelos_ha"][TIPO_ESTRATO_DISENO].items()}

print("Areas de estratos cargadas (ha):", {k: round(v, 2) for k, v in sorted(STRATUM_AREAS_HA.items())})

In [ ]:
# 2. Cruce instantaneo (< 1 s) de coordenadas EPSG:4326 de CEO contra los 4 mapas GeoTIFF locales
assert CEO_CSV_PATH.is_file(), (
    f"No se encontro el archivo validado de CEO en: {CEO_CSV_PATH}\n"
    "Coloca ahi el CSV exportado tras tu fotointerpretacion Sentinel-2 + Planet en Collect Earth Online."
)

df_ceo = pd.read_csv(CEO_CSV_PATH)
# Deteccion flexible de nombres de columnas tipicos de CEO (LON/lon, LAT/lat)
col_lon_use = next((c for c in [COL_LON, "LON", "lon", "center_lon"] if c in df_ceo.columns), COL_LON)
col_lat_use = next((c for c in [COL_LAT, "LAT", "lat", "center_lat"] if c in df_ceo.columns), COL_LAT)
col_str_use = next((c for c in [COL_STRATUM, "stratum", "stratum_id", "pl_stratum"] if c in df_ceo.columns), COL_STRATUM)

df_ceo["stratum_id"] = df_ceo[col_str_use].astype(int)
df_ceo["ref_id"] = df_ceo[COL_REF].map(
    lambda v: MAPEO_ETIQUETAS_CEO.get(v, MAPEO_ETIQUETAS_CEO.get(str(v).strip(), np.nan))
)
n_sin_mapear = int(df_ceo["ref_id"].isna().sum())
if n_sin_mapear > 0:
    print("Etiquetas sin mapear encontradas:", df_ceo.loc[df_ceo["ref_id"].isna(), COL_REF].unique())
    raise ValueError("Agrega las etiquetas faltantes a MAPEO_ETIQUETAS_CEO en la celda 1.")
df_ceo["ref_id"] = df_ceo["ref_id"].astype(int)

modelos_evaluar = ["MS", "SAR", "MS_SAR_PCA", "MS_SAR_FULL"]
map_paths = {m: Path(areas_json["map_paths"][m]) for m in modelos_evaluar}

df_val = cross_points_with_local_maps(df_ceo, col_lon_use, col_lat_use, map_paths)
print(f"Puntos CEO totales: {len(df_ceo)} | Puntos validos cruzados contra los 4 GeoTIFFs: {len(df_val)}")
display(df_val.groupby("stratum_id").size().rename("n_muestras_estrato").to_frame())
df_val.to_csv(dirs["tablas"] / f"ceo_cruzado_4modelos_{reg_l}.csv", index=False)

In [ ]:
# 3. Calculo de precisiones insesgadas (Olofsson 2014 / Stehman 2014), F1-Score ± IC 95% y Area Ajustada (ha)
resultados_olof = {}
resumen_oe3 = []
conf = float(val_cfg["nivel_confianza"])
n_boot = int(val_cfg["bootstrap_iteraciones"])
seed = int(cfg["proyecto"]["seed"])

for m_name in modelos_evaluar:
    res = stehman_generalized_accuracy(
        df=df_val,
        stratum_col="stratum_id",
        pred_col=f"pred_{m_name}",
        ref_col="ref_id",
        stratum_areas_ha=STRATUM_AREAS_HA,
        class_labels=CLASS_IDS,
        class_names=CLASES_NOMBRES,
        confidence=conf,
        n_bootstrap=n_boot,
        seed=seed,
    )
    resultados_olof[m_name] = res
    m_row = res["per_class"].set_index("clase_id").loc[1]

    resumen_oe3.append({
        "Modelo": m_name,
        "N_CEO": res["n_muestras"],
        "OA": res["OA"],
        "OA_CI_half": res["OA_CI_half"],
        "Kappa": res["Kappa"],
        "Kappa_CI_half": 0.5 * (res["Kappa_CI_high"] - res["Kappa_CI_low"]),
        "PU_Mangle": m_row["PU"],
        "PU_Mangle_CI_half": m_row["PU_CI_half"],
        "PP_Mangle": m_row["PP"],
        "PP_Mangle_CI_half": m_row["PP_CI_half"],
        "F1_Mangle": m_row["F1"],
        "F1_Mangle_CI_half": 0.5 * (m_row["F1_CI_high"] - m_row["F1_CI_low"]),
        "Area_Mapeada_Mangle_ha": areas_json["areas_modelos_ha"][m_name].get("1", np.nan),
        "Area_Ajustada_Mangle_ha": m_row["area_ajustada_ha"],
        "Area_Ajustada_CI_half_ha": m_row["area_CI_half_ha"],
    })

df_oe3 = pd.DataFrame(resumen_oe3)
print(f"=== RESULTADOS OFICIALES OLOFSSON/STEHMAN (IC {int(conf*100)}%) — {REGION} ===")
display(df_oe3)

In [ ]:
# 4. Prueba formal de Hipotesis (H0 vs H1): McNemar, Prueba Z ponderada por diseno, Z de Kappa y ΔF1 Bootstrap
alpha = float(val_cfg["alpha_hipotesis"])
pares = [
    ("MS_SAR_PCA", "MS"),
    ("MS_SAR_PCA", "SAR"),
    ("MS_SAR_FULL", "MS"),
    ("MS_SAR_FULL", "SAR"),
    ("MS_SAR_PCA", "MS_SAR_FULL"),
]

filas_hip = []
for mod_a, mod_b in pares:
    mc_all = mcnemar_test(df_val["ref_id"], df_val[f"pred_{mod_a}"], df_val[f"pred_{mod_b}"], alpha=alpha)
    mc_mgl = mcnemar_test(df_val["ref_id"], df_val[f"pred_{mod_a}"], df_val[f"pred_{mod_b}"], binary_target_class=1, alpha=alpha)
    z_all = stratified_paired_z_test(df_val, "stratum_id", f"pred_{mod_a}", f"pred_{mod_b}", "ref_id", STRATUM_AREAS_HA, alpha=alpha)
    z_mgl = stratified_paired_z_test(df_val, "stratum_id", f"pred_{mod_a}", f"pred_{mod_b}", "ref_id", STRATUM_AREAS_HA, binary_target_class=1, alpha=alpha)
    kz = kappa_z_test(
        resultados_olof[mod_a]["Kappa"], resultados_olof[mod_a]["Kappa_SE"],
        resultados_olof[mod_b]["Kappa"], resultados_olof[mod_b]["Kappa_SE"],
        alpha=alpha,
    )
    diff_f1 = resultados_olof[mod_a]["bootstrap_f1"][:, 0] - resultados_olof[mod_b]["bootstrap_f1"][:, 0]
    f1_ci_lo, f1_ci_hi = np.nanpercentile(diff_f1, [2.5, 97.5])

    sig = z_all["rechaza_H0"] or mc_mgl["rechaza_H0"]
    decision = ("Rechaza H0 (A > B significativo)" if z_all["delta_OA_A_menos_B"] > 0 else "Rechaza H0 (A < B significativo)") if sig else "No rechaza H0"
    filas_hip.append({
        "Comparacion": f"{mod_a} vs {mod_b}",
        "Delta_OA_pct": z_all["delta_OA_A_menos_B"] * 100.0,
        "Z_diseno_OA": z_all["Z_stat"],
        "p_Z_OA": z_all["p_value"],
        "p_McNemar_multiclase": mc_all["p_value_recomendado"],
        "p_McNemar_Mangle": mc_mgl["p_value_recomendado"],
        "p_Z_diseno_Mangle": z_mgl["p_value"],
        "Delta_Kappa": kz["delta_kappa"],
        "p_Z_Kappa": kz["p_value"],
        "Delta_F1_Mangle_IC95": f"[{f1_ci_lo*100:+.2f}%, {f1_ci_hi*100:+.2f}%]",
        "Decision_H0_alpha005": decision,
    })

df_hipotesis = pd.DataFrame(filas_hip)
print("=== CONTRASTE FORMAL DE HIPOTESIS (H0 vs H1) ===")
display(df_hipotesis)

In [ ]:
# 5. Figuras finales de tesis (300 DPI) y libro Excel maestro del Objetivo 3
reg_nom = cfg["regiones"][REGION]["nombre_largo"]

plot_model_comparison(
    df_oe3,
    f"Comparación de Modelos (Validación Independiente Olofsson / CEO, IC 95%) — {reg_nom}",
    save_path=dirs["figuras"] / f"OE3_fig1_comparacion_modelos_olofsson_{reg_l}.png",
)
plt.show()

for m_name in modelos_evaluar:
    plot_confusion_matrix(
        resultados_olof[m_name]["matrix_proportions"],
        f"Matriz de Confusión en Proporciones de Área (Olofsson) — {m_name} ({reg_nom})",
        is_proportion=True,
        save_path=dirs["figuras"] / f"OE3_cm_proporciones_{m_name.lower()}_{reg_l}.png",
    )
    plt.show()

fig_area, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(df_oe3))
w = 0.35
ax.bar(x - w/2, df_oe3["Area_Mapeada_Mangle_ha"], width=w, label="Área mapeada (píxeles)", color="#78c679", edgecolor="#222")
ax.bar(
    x + w/2, df_oe3["Area_Ajustada_Mangle_ha"], width=w,
    yerr=df_oe3["Area_Ajustada_CI_half_ha"], capsize=5,
    label="Área ajustada insesgada (Olofsson ± IC 95%)", color="#006837", edgecolor="#222",
)
marn_base = float(cfg["regiones"][REGION]["linea_base_marn_2013_ha"])
ax.axhline(marn_base, color="#d7191c", linestyle="--", linewidth=1.4, label=f"Referencia MARN (2013): {marn_base:,.1f} ha")
ax.set_xticks(x)
ax.set_xticklabels(df_oe3["Modelo"], fontweight="bold")
ax.set_ylabel("Superficie de bosque manglar (ha)")
ax.set_title(f"Estimación de Superficie de Manglar e Incertidumbre (IC 95%) — {reg_nom}", fontweight="bold")
ax.legend(frameon=True)
fig_area.tight_layout()
fig_area.savefig(dirs["figuras"] / f"OE3_fig2_area_manglar_olofsson_{reg_l}.png", dpi=300)
plt.show()

xlsx_oe3 = dirs["tablas"] / f"OE3_Validacion_Olofsson_Hipotesis_{reg_l}.xlsx"
with pd.ExcelWriter(xlsx_oe3, engine="openpyxl") as writer:
    df_oe3.to_excel(writer, sheet_name="Resumen_Modelos_OE3", index=False)
    df_hipotesis.to_excel(writer, sheet_name="Prueba_Hipotesis", index=False)
    for m_name in modelos_evaluar:
        resultados_olof[m_name]["per_class"].to_excel(writer, sheet_name=f"Metricas_{m_name}", index=False)
        resultados_olof[m_name]["matrix_proportions"].to_excel(writer, sheet_name=f"CM_Prop_{m_name}")
        resultados_olof[m_name]["matrix_counts"].to_excel(writer, sheet_name=f"CM_Cont_{m_name}")

print(f"[OK] Entregables finales exportados en: {xlsx_oe3}")
print(f"[OK] Figuras 300 DPI guardadas en     : {dirs['figuras']}")